<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 LangGraph 101 — 本节精华

**一句话定位**:这是整门 Ambient Agents 课的 **地基** —— 学完后你能用 LangChain + LangGraph + LangSmith **三件套** 搭出最基础的工作流和 agent。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**这节课在整门课的位置**:Module 1 / 6 —— **打地基**。后续 Module 2(Build)、Module 3(Eval)、Module 4(HITL)、Module 5(Memory)、Module 6(Deploy)全部建立在这节课的 5 个原语之上。

**5 个原语**:`init_chat_model` · `@tool` + `bind_tools` · `StateGraph(State, Node, Edge)` · `checkpointer + thread_id` · `interrupt + Command`。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🗺️ 11 个知识点鸟瞰

</div>

| # | 知识点 | 一句话价值 | 后面哪节会用 |
|---|--------|------------|-------------|
| 1 | `init_chat_model` | 一行换 provider(OpenAI / Anthropic / ...) | 所有后续课 |
| 2 | `.invoke()` / `.stream()` | 调模型的标准方法 | 所有后续课 |
| 3 | `@tool` 装饰器 | Python 函数 → 工具,自动推断名/描述/参数 | M2 Build |
| 4 | `bind_tools(...)` + `tool_choice` | 把工具挂给 LLM,可强制调用 | M2 Build |
| 5 | **Workflow** 概念 | LLM 嵌进**预定义路径**(可在白板画清) | M2 Triage Router |
| 6 | **Agent** 概念 | LLM **自己决定** 工具调用顺序,循环到结束 | M2 Response Agent |
| 7 | `StateGraph` 三件套 | **State** + **Node** + **Edge** = LangGraph 全部底层 | M2 - M6 |
| 8 | `MessagesState` + 条件边 | 预制 state(自动 append)+ 分支路由 | M2 + M4 |
| 9 | `create_react_agent` | 一行搭好 ReAct 循环 | M2 起步 |
| 10 | **Checkpointer + thread_id** | 持久化 state,支持多轮对话 | M4 + M5 |
| 11 | **`interrupt` + `Command(resume=...)`** | HITL 的物理基础 | M4 重点 |

**额外送你两个**:LangSmith **环境变量自动 trace**、LangGraph Platform **`langgraph dev`** 本地部署。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ① Chat Models · Tools · Tool Calling — 三件套

</div>

**LLM 调工具的 3 步**:`@tool` 把函数变工具 → `bind_tools` 挂给 LLM → LLM 输出 **结构化 tool_calls**(不是真调,是参数) → 你拿这些参数手动调工具。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from langchain.chat_models import init_chat_model
from langchain.tools import tool

llm = init_chat_model("openai:gpt-4o")

@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    return f"Email sent to {to}"

# 关键一步:挂工具
llm_with_tools = llm.bind_tools(
    [write_email],
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">tool_choice="any"</span>,         # 强制调用任意一个工具
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">parallel_tool_calls=False</span>  # 一次只调一个
)

output = llm_with_tools.invoke("Email my boss...")
# output.tool_calls = [{"name":"write_email", "args":{...}, "id":"..."}]
</code></pre>

**记住**:`bind_tools` 不会让 LLM **真的** 执行工具——它只让 LLM **学会输出符合 tool 签名的 JSON 参数**。执行还得你自己来(或者交给 `ToolNode`)。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:`tool_choice` 和 `parallel_tool_calls` 看似细节,实则是 **生产级控制**。

- `tool_choice="any"`:LLM **必须**调一个工具(不能纯文字回复)—— 用在「**必须要执行动作**」的场景
- `tool_choice="none"`:**禁止**调工具 —— 用在「**纯生成文本**」的场景
- `parallel_tool_calls=False`:LLM 一次只产 **1 个 tool_call** —— 更可控,适合 HITL

默认值是「LLM 自己决定」,大多数 prod 场景里你都需要 **显式锁死**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ② Workflow vs Agent — 用哪个的判断准则

</div>

| 维度 | 🔵 Workflow | 🔴 Agent |
|------|-------------|----------|
| 控制流 | **你预先画好** 流程图 | LLM **运行时自己决定** |
| Agency(自主性) | 低 | 高 |
| Predictability(可预测) | 高 | 低 |
| 延迟 / 成本 | 通常低 | 通常高(多轮 LLM) |
| 适用场景 | 流程**能在白板上画清** | 流程**到运行时才知道** |
| 典型例子 | 「先 triage,再回信」 | 「这封邮件该查日历?约会?直接回?」 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**Lance 的口诀**:**"If you can easily draw the control flow on a whiteboard, a workflow is appropriate."**

👉 这正是 Module 2 的核心设计:**外层 workflow(triage router 写死)+ 内层 agent(response agent 自由发挥)**。LangGraph 的杀手锏就是 **能把两者拼在一张图里**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ③ LangGraph 三件套 — State · Node · Edge

</div>

| 概念 | 是什么 | Python 中长什么样 |
|------|--------|-------------------|
| **State** | 你想在整个图里追踪的信息 schema | `TypedDict` / `dataclass` / `Pydantic` |
| **Node** | 「怎么更新 state」的工作单元 | 普通 Python 函数,签名 `(state) -> dict` |
| **Edge** | 「下一步去哪个 node」的连接 | `.add_edge("a", "b")` 或条件边函数 |

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class StateSchema(TypedDict):
    request: str
    email: str

def write_email_node(state: StateSchema) -> dict:
    output = llm_with_tools.invoke(state["request"])
    args = output.tool_calls[0]["args"]
    email = write_email.invoke(args)
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">return {"email": email}</span>   # ← 只返回想更新的字段

workflow = StateGraph(StateSchema)
workflow.add_node("write_email_node", write_email_node)
workflow.add_edge(START, "write_email_node")
workflow.add_edge("write_email_node", END)
app = workflow.compile()
</code></pre>

**节点返回 dict 的含义**:更新 state 中对应的 key。**默认覆盖**;若要 append,需要在 state schema 上声明 `Annotated[..., reducer]`(或直接用 `MessagesState`)。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察:reducer 是 state 更新的「黏合剂」**

- **默认行为**:节点返回 `{"email": "..."}` → state 里 email 字段 **被整个覆盖**
- **MessagesState** 的 messages 字段绑了 `add_messages` reducer → 节点每次返回的 message **被 append 进列表**

为什么这事很重要?**agent 循环需要消息累积**,所以建 agent 时几乎一定用 `MessagesState` 或自己挂 reducer。`add` reducer 就是 LangGraph 后面所有「记忆」机制的雏形。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ④ 条件边 + create_react_agent — 一行搭 agent

</div>

**条件边**:节点返回 dict 是更新 state,**条件边函数返回字符串是「下一个 node 的名字」**。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from typing import Literal
from langgraph.graph import MessagesState

def should_continue(state: MessagesState) -> <span style="background:#3d3414; color:#fde047; padding:0 2px;">Literal["run_tool", END]</span>:
    last = state["messages"][-1]
    if last.tool_calls:
        return "run_tool"   # ← 这就是下一个 node 的名字
    return END

workflow.add_conditional_edges("call_llm", should_continue)
</code></pre>

**Agent = LLM ↔ tool 循环 + 上面这条 should_continue 边**。手写一遍是给你「祛魅」用的,平时直接用预制:

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=[write_email],
    prompt="You are an email assistant."
)
agent.invoke({"messages": [{"role": "user", "content": "..."}]})
</code></pre>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ⑤ Persistence — Threads + Interrupts(本课**最重要**)

</div>

**Checkpointer** 在每个 node 之后存一份 state 快照,绑到一个 `thread_id`。同一 thread 再次 invoke 时,**state 自动复现**——这就是多轮对话的物理基础。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from langgraph.checkpoint.memory import InMemorySaver

agent = create_react_agent(llm, tools, checkpointer=<span style="background:#3d3414; color:#fde047; padding:0 2px;">InMemorySaver()</span>)

config = {"configurable": {"thread_id": "user-42"}}
agent.invoke({"messages": [...]}, config=config)   # 第 1 轮
agent.invoke({"messages": [...]}, config=config)   # 第 2 轮,自动接着上文

# 任意时刻取整个 state:
state = agent.get_state(config)
</code></pre>

**Interrupt** 让图在指定节点**暂停**,等用户输入后用 `Command(resume=...)` **续跑**。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from langgraph.types import interrupt, Command

def human_feedback(state):
    user_input = <span style="background:#3d3414; color:#fde047; padding:0 2px;">interrupt("please provide feedback")</span>   # 图在这停
    return {"user_feedback": user_input}

# 续跑
graph.invoke(<span style="background:#3d3414; color:#fde047; padding:0 2px;">Command(resume="looks good!")</span>, config=config)
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察:`interrupt` + `Command(resume=...)` 是 HITL 的物理基础**

整个 Module 4 的 Agent Inbox 就是构建在这一对原语之上的——所有「在敏感 tool call 前等用户确认」的功能,本质都是:

1. 节点内调 `interrupt(payload)` → graph 暂停 → payload 显示给用户
2. 用户做选择 → 前端 / SDK 发 `Command(resume={...})` → graph 续跑

传给 `interrupt` 的 payload 决定 Agent Inbox 渲染什么(action / config / description)。**学会这一对,后面 HITL 就是写胶水**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ⑥ Tracing + Deployment — 让 agent 跑出 notebook

</div>

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-radius:4px; border-left:4px solid #22d3ee; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🔭 LangSmith Tracing — 设两个环境变量就行**

```bash
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY="<your-key>"
```

所有 LangChain / LangGraph 调用 **自动 trace**。在 LangSmith UI 里能看到每个 node、每次 LLM 调用、bound tools、tool_call、ToolMessage、final AIMessage —— **零额外代码**。

**🚀 LangGraph Platform — `langgraph dev` 本地部署**

项目根目录放 `langgraph.json`(配置文件)+ 图文件,跑 `langgraph dev` 自动起服务 + 开 **LangGraph Studio**(可视化 IDE)。

```json
{
  "graphs": { "langgraph101": "./src/email_assistant/langgraph_101.py:app" },
  "env": ".env",
  "dependencies": ["."]
}
```

- **本地** 部署:checkpoints 写本地文件,免费,适合开发
- **托管** 部署(LangGraph Platform Cloud):Postgres 存 checkpoint,适合生产

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎯 学完这节,你能做什么了

</div>

| 能力 | 用到的 API |
|------|-----------|
| 调用任意 LLM provider | `init_chat_model("provider:model")` |
| 把 Python 函数变成 LLM 可调工具 | `@tool` |
| 让 LLM 输出符合工具签名的参数 | `llm.bind_tools([...], tool_choice=...)` |
| 画一个 workflow(节点+边的图) | `StateGraph(...) + add_node + add_edge` |
| 加条件分支 | `add_conditional_edges(node, fn)` |
| 让 messages 自动累积 | 用 `MessagesState` |
| 一行起 ReAct agent | `create_react_agent(model, tools, prompt)` |
| 支持多轮对话 | `compile(checkpointer=...)` + `thread_id` |
| 让图暂停等用户决定 | `interrupt(payload)` |
| 续跑暂停的图 | `Command(resume=...)` |
| 自动 trace 到 LangSmith | 设 `LANGSMITH_TRACING=true` |
| 把 notebook 变成可部署服务 | `langgraph.json` + `langgraph dev` |

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 常见误解,提前避雷**

- ❌ 以为 `bind_tools` 会让 LLM 真调工具 —— 它只让 LLM 输出 **参数 JSON**,执行还得你来
- ❌ 以为节点 return 整个新 state —— 实际只 return **想更新的字段**(其他保持原样)
- ❌ 以为 MessagesState 跟普通 dict 一样会覆盖 —— 它装了 `add_messages` reducer,**自动 append**
- ❌ 以为 interrupt 之后 graph 死了 —— 它只是 **暂停**,state 在 checkpoint 里,`Command(resume=...)` 一调就复活
- ❌ 以为部署要写 server / Dockerfile —— `langgraph.json` + `langgraph dev` 就够了

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

LangGraph 的全部魔法 = **`State` + `Node` + `Edge`** 三件套 **+** **`checkpointer`(记忆)+ `interrupt`(暂停)+ `Command`(续跑)** 三件套。

**这 6 样东西** 就是后面 Build / Eval / HITL / Memory / Deploy 全部 5 个模块的 **物理基础**——后面无非是把这套基础元件**组合搭配**用在邮件场景。

📎 配套地图:[../_课程总览_🏞️.svg](../_课程总览_🏞️.svg) · 主课:[./0_langgraph_101.ipynb](./0_langgraph_101.ipynb)

</div>